# 05 — Predicción del ticket medio diario

## Objetivo
Predecir el ticket medio diario.

La variable objetivo será el ticket medio por día, construido a partir de la facturación total y el número de tickets.


## Objetivo
Predecir el ticket medio diario usando un flujo estándar de machine learning con separación train/test temporal, feature engineering y comparación de modelos.

En este notebook se construye un dataset diario con:
- facturación total
- número de tickets
- ticket medio
- variables temporales y de historia reciente

La idea es entrenar modelos de regresión para predecir el ticket medio del próximo día o de días futuros.


In [ ]:
# 1) Imports y configuración
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor

try:
    import xgboost as xgb
except Exception:
    xgb = None

SEED = 42
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data'
SILVER_SNAP = DATA_DIR / 'silver' / 'snapshots'
RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('talk')

print(f'Project root: {PROJECT_ROOT}')
print(f'Silver snapshots: {SILVER_SNAP}')
print(f'Results dir: {RESULTS_DIR}')


In [ ]:
# 2) Cargar los datos depurados y preparar el dataset diario

# Cargamos los datos de Silver. Si el proyecto tiene diferente nombre de archivo,
# se usa el fallback por compatibilidad.
try:
    df = pd.read_parquet(SILVER_SNAP / 'tickets_silver.parquet')
except FileNotFoundError:
    df = pd.read_parquet(SILVER_SNAP / 'facturas_silver.parquet')

print('shape inicial:', df.shape)
print(df.head(2).to_dict(orient='records'))

# Normalizamos la columna de fecha
fecha_col = None
for col in ['date', 'fecha', 'datetime', 'created_at', 'document_date']:
    if col in df.columns:
        fecha_col = col
        break

if fecha_col is None:
    raise KeyError('No se encontró una columna de fecha en los datos de Silver.')

df['fecha'] = pd.to_datetime(df[fecha_col])

# Usamos facturación y número de tickets como base para construir ticket medio diario
# Si las columnas no existen, se reutiliza el nombre equivalente de negocio.
if 'document_total' in df.columns:
    value_col = 'document_total'
elif 'total' in df.columns:
    value_col = 'total'
else:
    raise KeyError('No existe una columna de importe para calcular la facturación.')

if 'document_id' in df.columns:
    id_col = 'document_id'
elif 'ticket_id' in df.columns:
    id_col = 'ticket_id'
elif 'id' in df.columns:
    id_col = 'id'
else:
    raise KeyError('No existe una columna identificadora de tickets/documentos.')

# Agregación diaria
agg = (
    df.groupby('fecha', as_index=False)
      .agg(
          facturacion=(value_col, 'sum'),
          num_tickets=(id_col, 'nunique')
      )
      .sort_values('fecha')
      .reset_index(drop=True)
)

agg['ticket_medio'] = agg['facturacion'] / agg['num_tickets']
print(agg.head())
print(agg.describe().round(2))


## Variables clave para este problema

Vamos a predecir el ticket medio diario usando una base temporal con:
- facturación total por día
- número de tickets por día
- ticket medio calculado como facturación / tickets
- lags históricos y medias móviles
- variables temporales: día de la semana, mes, fin de semana
- variables de negocio y clima si están disponibles en Silver


In [ ]:
# 3) Construir la serie diaria y la variable objetivo
# En este caso la variable objetivo es el ticket medio diario: facturacion / num_tickets

# Si el proyecto ya tiene una tabla diaria, podrías cargarla directamente.
# Aquí se reusa el patron de agregación desde los tickets de Silver.

daily = (
    df.assign(fecha=pd.to_datetime(df['fecha']))
      .groupby('fecha', as_index=False)
      .agg(
          facturacion=(value_col, 'sum'),
          num_tickets=(id_col, 'nunique')
      )
      .sort_values('fecha')
      .reset_index(drop=True)
)

daily['ticket_medio'] = daily['facturacion'] / daily['num_tickets']

print(daily.head())
print(daily[['facturacion', 'num_tickets', 'ticket_medio']].describe().round(2))


In [ ]:
# 4) Feature engineering temporal e histórico

daily['dia_num'] = daily['fecha'].dt.dayofweek
daily['mes'] = daily['fecha'].dt.month
daily['es_fin_semana'] = daily['fecha'].dt.dayofweek.isin([5, 6]).astype(int)
daily['es_lunes'] = (daily['dia_num'] == 0).astype(int)

daily = daily.sort_values('fecha').reset_index(drop=True)

for lag in [1, 3, 7, 14, 30]:
    daily[f'lag_{lag}_ticket'] = daily['ticket_medio'].shift(lag)

for window in [3, 7, 14, 30]:
    daily[f'media_movil_{window}_ticket'] = daily['ticket_medio'].shift(1).rolling(window, min_periods=1).mean()

# Opción adicional: usar la facturación y número de tickets para añadir contexto
for lag in [1, 7, 14]:
    daily[f'lag_{lag}_facturacion'] = daily['facturacion'].shift(lag)

for lag in [1, 7, 14]:
    daily[f'lag_{lag}_tickets'] = daily['num_tickets'].shift(lag)

daily = daily.dropna().reset_index(drop=True)
print(daily.head())
print(f'Filas finales: {len(daily)}')


In [ ]:
# 5) Preparar X e y para modelado

feature_cols = [
    'dia_num', 'mes', 'es_fin_semana', 'es_lunes',
    'lag_1_ticket', 'lag_3_ticket', 'lag_7_ticket', 'lag_14_ticket', 'lag_30_ticket',
    'media_movil_3_ticket', 'media_movil_7_ticket', 'media_movil_14_ticket', 'media_movil_30_ticket',
    'lag_1_facturacion', 'lag_7_facturacion', 'lag_14_facturacion',
    'lag_1_tickets', 'lag_7_tickets', 'lag_14_tickets'
]

X = daily[feature_cols].copy()
y = daily['ticket_medio'].copy()

print('Shape X:', X.shape)
print('Shape y:', y.shape)
print(X.head())


In [ ]:
# 6) Split temporal train/test
split_idx = int(len(daily) * 0.8)
X_train, X_test = X.iloc[:split_idx].copy(), X.iloc[split_idx:].copy()
y_train, y_test = y.iloc[:split_idx].copy(), y.iloc[split_idx:].copy()

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('Train target range:', round(y_train.min(), 2), '->', round(y_train.max(), 2))
print('Test target range:', round(y_test.min(), 2), '->', round(y_test.max(), 2))


In [ ]:
# 7) Define modelos básicos de regresión
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor

models = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'RandomForest': RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=2,
        random_state=SEED,
        n_jobs=-1,
    ),
    'ExtraTrees': ExtraTreesRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=2,
        random_state=SEED,
        n_jobs=-1,
    ),
    'HistGradientBoosting': HistGradientBoostingRegressor(
        max_depth=None,
        learning_rate=0.05,
        max_iter=400,
        random_state=SEED,
    ),
}

if xgb is not None:
    models['XGBoost'] = xgb.XGBRegressor(
        objective='reg:squarederror',
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=SEED,
        n_jobs=-1,
    )

print('Modelos definidos:', list(models.keys()))


In [ ]:
# 8) Pipeline y comparación de modelos
# Usamos un pipeline simple con imputación y escalado sólo para modelos lineales.
# Para árboles se puede dejar tal cual porque no requieren escalado.

results = []

for name, model in models.items():
    # Seleccionamos columnas numéricas para el pipeline; en este caso todas son numéricas
    numeric_features = X_train.columns.tolist()
    preprocess = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    pipeline = Pipeline([
        ('preprocess', preprocess),
        ('model', model),
    ])

    pipeline.fit(X_train[numeric_features], y_train)
    preds = pipeline.predict(X_test[numeric_features])

    mae = mean_absolute_error(y_test, preds)
    rmse = mean_squared_error(y_test, preds, squared=False)
    r2 = r2_score(y_test, preds)

    results.append({
        'model': name,
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2,
    })

    print(f'\n=== {name} ===')
    print(f'MAE: {mae:.3f}')
    print(f'RMSE: {rmse:.3f}')
    print(f'R2: {r2:.3f}')

results_df = pd.DataFrame(results).sort_values('MAE', ascending=True)
print('\nRanking por MAE:')
print(results_df.round(4).to_string(index=False))
